In [1]:
import pandas as pd

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sid321axn/malicious-urls-dataset")

print("Path to dataset files:", path)

Path to dataset files: /Users/maciasalvasalva/.cache/kagglehub/datasets/sid321axn/malicious-urls-dataset/versions/1


In [3]:
df = pd.read_csv(f"{path}/malicious_phish.csv")

In [4]:
df.head()

,url,type
0,br-icloud.com.br,phishing
1,mp3raid.com/music/krizz_kaliko.html,benign
2,bopsecrets.org/rexroth/cr/1.htm,benign
3,http://www.garage-pirenne.be/index.php?option=...,defacement
4,http://adventure-nicaragua.net/index.php?optio...,defacement


In [5]:
df['type'].value_counts()

type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64

Convert url to domains

In [6]:
from urllib.parse import urlparse
import pandas as pd
import re

def get_domain(url):
    if pd.isna(url):
        return None
    
    url = str(url).strip()

    # Si empieza con // (a veces pasa)
    if url.startswith("//"):
        url = "http:" + url

    # Si no tiene esquema, agregamos http://
    if not url.startswith(("http://", "https://")):
        url = "http://" + url

    try:
        parsed = urlparse(url)
    except ValueError:
        return None   # para URLs rotas tipo IPv6 incorrectas

    domain = parsed.netloc.lower()

    # Quitar user@
    if "@" in domain:
        domain = domain.split("@")[-1]

    # IPv6: vienen entre []
    if domain.startswith("[") and domain.endswith("]"):
        domain = domain[1:-1]
        return domain

    # Quitar puertos
    if ":" in domain:
        domain = domain.split(":")[0]

    # Si quedó vacío, descartar
    if not domain:
        return None
    
    return domain

In [7]:
df["domain"] = df["url"].apply(get_domain)

In [8]:
df_unique = df[['domain', 'type']].drop_duplicates(subset='domain').reset_index(drop=True)

In [9]:
df_unique.head(20)

,domain,type
0,br-icloud.com.br,phishing
1,mp3raid.com,benign
2,bopsecrets.org,benign
3,www.garage-pirenne.be,defacement
4,adventure-nicaragua.net,defacement
5,buzzfil.net,benign
6,espn.go.com,benign
7,yourbittorrent.com,benign
8,www.pashminaonline.com,defacement
9,allmusic.com,benign


In [10]:
df_unique['type'].value_counts()

type
benign        131762
phishing       55291
malware         7345
defacement      2122
Name: count, dtype: int64

In [11]:
df_finall = df_unique[(df_unique['type'] == 'phishing') | (df_unique['type'] == 'malware') | (df_unique['type'] == 'benign')]

In [12]:
df_finall['type'].value_counts()

type
benign      131762
phishing     55291
malware       7345
Name: count, dtype: int64

In [13]:
df_finall_whitelist = df_finall[df_finall['type'] == "benign"]
df_finall_blacklist = df_finall[(df_finall['type'] == 'malware') | (df_finall['type'] == 'benign')]

In [14]:
df_finall_whitelist['domain'].to_csv('extra_whitelist.csv', index=False, header=False)

In [16]:
df_finall_whitelist.info()

<class 'pandas.core.frame.DataFrame'>
Index: 131762 entries, 1 to 174859
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   domain  131762 non-null  object
 1   type    131762 non-null  object
dtypes: object(2)
memory usage: 3.0+ MB


In [15]:
df_finall_blacklist['domain'].to_csv('extra_blacklist.csv', index=False, header=False)

In [17]:
df_finall_blacklist.info()

<class 'pandas.core.frame.DataFrame'>
Index: 139107 entries, 1 to 174859
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   domain  139107 non-null  object
 1   type    139107 non-null  object
dtypes: object(2)
memory usage: 3.2+ MB
